<a href="https://colab.research.google.com/github/dehande/Project_2_regression-benchmark/blob/main/Proje2_Complex_Regression_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving regression_playground.csv to regression_playground.csv


In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error

# 1) Veriyi oku
df = pd.read_csv("regression_playground.csv")
X = df[["age"]]
y = df["salary"]

# 2) Split'i sabitle (adil kıyas için)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3) Aynı polynomial dereceyi kullanacağımız bir fonksiyon yazalım (tekrar kod yazmamak için)
def prepare_features(degree: int):
  """Verilen degree için X_train/X_test'i polynomial + scaler ile hazırlar."""
  poly = PolynomialFeatures(degree=degree, include_bias=False)
  X_train_poly = poly.fit_transform(X_train)
  X_test_poly = poly.transform(X_test)

  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train_poly)
  X_test_scaled = scaler.transform(X_test_poly)

  feature_names = poly.get_feature_names_out(["age"])
  return X_train_scaled, X_test_scaled, feature_names

rows = []

# --- Model 1: Baseline Linear (degree=1) ---
Xtr1, Xte1, names1 = prepare_features(degree=1)
lin = LinearRegression()
lin.fit(Xtr1, y_train)
pred_tr = lin.predict(Xtr1)
pred_te = lin.predict(Xte1)
rows.append({
    "model": "linear(deg=1)",
    "param": "-",
    "train_mae": mean_absolute_error(y_train, pred_tr),
    "test_mae": mean_absolute_error(y_test, pred_te),
    "gap(test-train)": mean_absolute_error(y_test, pred_te) - mean_absolute_error(y_train, pred_tr),
    "zero_coef": np.nan
})

# --- Model 2: Polynomial Linear (degree=5) ---
degree_poly = 5
XtrP, XteP, nameP = prepare_features(degree=degree_poly)
poly_lr = LinearRegression()
poly_lr.fit(XtrP, y_train)
pred_tr = poly_lr.predict(XtrP)
pred_te = poly_lr.predict(XteP)
rows.append({
    "model": f"poly_lr(deg={degree_poly})",
    "param": "-",
    "train_mae": mean_absolute_error(y_train, pred_tr),
    "test_mae": mean_absolute_error(y_test, pred_te),
    "gap(test-train)": mean_absolute_error(y_test, pred_te) - mean_absolute_error(y_train, pred_tr),
    "zero_coef": np.nan
})

# --- Model 3: Ridge (degree=5, alpha=0.5) ---
alpha_ridge = 0.5
ridge = Ridge(alpha=alpha_ridge)
ridge.fit(XtrP, y_train)
pred_tr = ridge.predict(XtrP)
pred_te = ridge.predict(XteP)
rows.append({
    "model": f"ridge(deg={degree_poly})",
    "param": f"alpha={alpha_ridge}",
    "train_mae": mean_absolute_error(y_train, pred_tr),
    "test_mae": mean_absolute_error(y_test, pred_te),
    "gap(test-train)": mean_absolute_error(y_test, pred_te) - mean_absolute_error(y_train, pred_tr),
    "zero_coef": np.nan
})

# --- Model 4: Lasso (degree=5, alpha=10) ---
alpha_lasso = 10
lasso = Lasso(alpha=alpha_lasso, max_iter=50000)
lasso.fit(XtrP, y_train)
pred_tr = lasso.predict(XtrP)
pred_te = lasso.predict(XteP)
zero_count = int(np.sum(lasso.coef_ == 0))
rows.append({
    "model": f"lasso(deg={degree_poly})",
    "param": f"alpha={alpha_lasso}",
    "train_mae": mean_absolute_error(y_train, pred_tr),
    "test_mae": mean_absolute_error(y_test, pred_te),
    "gap(test-train)": mean_absolute_error(y_test, pred_te) - mean_absolute_error(y_train, pred_tr),
    "zero_coef": zero_count
})

# 4) Sonuç tablosu: test_mae en iyi olan üste gelsin
result = pd.DataFrame(rows).sort_values("test_mae")
print(result)

# 5) En iyi satırı otomatik seç
best = result.iloc[0]
print("\nBest Model->", best["model"], "|", best["param"], "|", "test_mae", best["test_mae"])

            model      param   train_mae    test_mae  gap(test-train)  \
0   linear(deg=1)          -  687.038848  903.758686       216.719837   
3    lasso(deg=5)   alpha=10  683.614796  907.806959       224.192163   
2    ridge(deg=5)  alpha=0.5  682.586839  910.335615       227.748777   
1  poly_lr(deg=5)          -  688.119931  925.503878       237.383947   

   zero_coef  
0        NaN  
3        3.0  
2        NaN  
1        NaN  

Best Model-> linear(deg=1) | - | test_mae 903.7586858761409
